<a href="https://colab.research.google.com/github/AI-Object-Dedection/sam3/blob/main/colab_faz4_egitim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAM3 Faz 4 — Tam Eğitim (Colab Pro+ / Arka Plan)

Bu notebook, Kaggle'da yaşanan iki sorunu çözmek için hazırlandı:
1. **Checkpoint kaybolması** → Checkpointler artık doğrudan **Google Drive**'a kaydediliyor
2. **Bağlantı kopunca eğitimin durması** → Eğitim **arka planda (background)** bir process olarak başlatılıyor. Colab Pro+'ın arka plan çalıştırma özelliği sayesinde sekmeyi kapatsan bile eğitim devam eder.

**ÖNEMLİ:** Çalıştırmadan önce `Runtime > Change runtime type > GPU` (Pro+'da A100/L4 seçebilirsin, T4 de çalışır) seç ve **Background execution**'ı aktif et (Runtime menüsünden veya Colab Pro+ ile otomatik gelir).

## Adım 0 — GPU Kontrolü

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU aktif: {torch.cuda.get_device_name(0)}")
    print(f"GPU bellek: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("UYARI: GPU bulunamadı! Runtime > Change runtime type > GPU seç.")

## Adım 1 — Google Drive Bağlantısı

Veri seti, kod ve checkpointler Drive'da `MyDrive/sam3/` altında olacak.
Çalıştırınca izin isteyecek, onayla.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = "/content/drive/MyDrive/sam3"

# Veri seti, kod, checkpoint ve log klasörleri Drive altında
DRIVE_DATA_DIR       = f"{DRIVE_ROOT}/dacl10k/"
DRIVE_CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints/"
DRIVE_LOG_DIR        = f"{DRIVE_ROOT}/logs/"
REPO_DIR             = f"{DRIVE_ROOT}/SAM3"

os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_LOG_DIR, exist_ok=True)

print("Drive bağlandı!")
print("Veri seti yolu  :", DRIVE_DATA_DIR)
print("Checkpoint yolu :", DRIVE_CHECKPOINT_DIR)
print("Log yolu        :", DRIVE_LOG_DIR)

## Adım 2 — Proje Kodunu Hazırla

Proje koduna **Drive üzerinden** erişiyoruz, böylece checkpointler ve loglar
Drive'a yazılırken kod da kalıcı bir yerde durur (her oturumda yeniden clone gerekmez).

İlk seferde repo Drive'a klonlanır. Sonraki seferlerde mevcut klasör güncellenir (`git pull`).

In [ ]:
REPO_URL = "https://github.com/AI-Object-Dedection/sam3.git"

if not os.path.isdir(REPO_DIR):
    print("Repo Drive'a klonlanıyor (ilk sefer)...")
    !git clone {REPO_URL} "{REPO_DIR}"
else:
    print("Repo zaten Drive'da, güncelleniyor...")
    !cd "{REPO_DIR}" && git pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt
# NOT: Colab'da gelen torchao surumu (0.10.0) peft ile uyumsuz
# (ImportError: Found an incompatible version of torchao). Guncelleme
# denemesi .so dosyalarini bozdu, bu yuzden torchao'yu tamamen kaldiriyoruz.
# peft, torchao yuklu degilse o kontrolu atlar ve LoRA normal sekilde uygulanir.
!pip uninstall -y -q torchao

## Adım 3 — HuggingFace Login

Sol menüde 🔑 Secrets'a `HF_TOKEN` ekle (Faz 3'te yaptığın gibi).

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("HuggingFace girişi başarılı!")

## Adım 4 — Veri Setini İndir (Drive'a)

Veri seti Drive'da `MyDrive/sam3/dacl10k/` altında değilse, HuggingFace'ten
doğrudan **Drive'a** indirilir. `SAM3_DATA_DIR` ortam değişkeni sayesinde
`scripts/download_dataset.py`, `data/dacl10k/` yerine Drive klasörünü kullanır.

Veri seti zaten Drive'daysa bu hücre hızlıca atlar (mevcut dosyalar tekrar indirilmez).

In [ ]:
env = os.environ.copy()
env["SAM3_DATA_DIR"] = DRIVE_DATA_DIR

!cd "{REPO_DIR}" && SAM3_DATA_DIR="{DRIVE_DATA_DIR}" python scripts/download_dataset.py

## Adım 5 — Veri Seti Kontrolü

Veri setinin Drive'da `MyDrive/sam3/dacl10k/` altında, `images/{train,validation}` ve
`annotations/{train,validation}` klasörleriyle olduğunu kontrol ediyoruz.

In [ ]:
train_img_dir = os.path.join(DRIVE_DATA_DIR, "images", "train")
val_img_dir   = os.path.join(DRIVE_DATA_DIR, "images", "validation")

if not os.path.isdir(train_img_dir):
    raise FileNotFoundError(
        f"Veri seti bulunamadı: {train_img_dir}\n"
        "DACL10K veri setini Drive'da 'MyDrive/sam3/dacl10k/' altına yükle."
    )

train_count = len([f for f in os.listdir(train_img_dir) if f.lower().endswith(".jpg")])
val_count   = len([f for f in os.listdir(val_img_dir)   if f.lower().endswith(".jpg")])
print(f"Eğitim görseli    : {train_count}")
print(f"Validation görseli: {val_count}")

## Adım 6 — Eğitimi Arka Planda Başlat

`scripts/run_train.py`, ortam değişkenleriyle Drive yollarını kullanacak şekilde
**arka plan process** olarak başlatılır. Hücre çalışması bittiğinde process **durmaz**,
Drive'a log yazmaya devam eder.

- `SAM3_DATA_DIR`       → veri seti Drive'dan okunur
- `SAM3_CHECKPOINT_DIR` → checkpointler Drive'a yazılır (her `CHECKPOINT_EVERY_STEPS` adımda + her epoch sonunda)
- `NUM_EPOCHS`          → kaç epoch eğitileceği

In [ ]:
import subprocess

env = os.environ.copy()
env["SAM3_DATA_DIR"] = DRIVE_DATA_DIR
env["SAM3_CHECKPOINT_DIR"] = DRIVE_CHECKPOINT_DIR
env["NUM_EPOCHS"] = "10"

log_path = os.path.join(DRIVE_LOG_DIR, "faz4_train.log")
log_file = open(log_path, "a")

process = subprocess.Popen(
    ["python", "scripts/run_train.py"],
    cwd=REPO_DIR,
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print(f"Eğitim arka planda başladı. PID: {process.pid}")
print(f"Log dosyası: {log_path}")
print("Bağlantı kopsa bile eğitim devam eder, Adım 6 ile takip edebilirsin.")

## Adım 7 — Eğitimi Takip Et

Bu hücreyi istediğin zaman tekrar çalıştırarak son logları görebilirsin
(eğitimi durdurmaz, sadece log dosyasının son satırlarını okur).

In [ ]:
log_path = "/content/drive/MyDrive/sam3/logs/faz4_train.log"
!tail -n 40 "{log_path}"

## Adım 7 — Checkpointleri Kontrol Et

Drive'daki checkpoint klasörünü listeler. `epoch_X_lora` ve `best_lora` klasörleri
eğitim ilerledikçe burada birikir.

In [ ]:
checkpoint_dir = "/content/drive/MyDrive/sam3/checkpoints/"
for name in sorted(os.listdir(checkpoint_dir)):
    print(name)

## Özet

- Veri seti, kod, checkpoint ve loglar Drive'da `MyDrive/sam3/` altında — kalıcı
- Eğitim arka plan process olarak çalışıyor — sekme/bağlantı kopması eğitimi durdurmaz
- `CHECKPOINT_EVERY_STEPS` (config.py) sayesinde epoch bitmeden de ara checkpoint alınıyor
- Eğitim bittiğinde Adım 7 ile checkpointleri, Adım 6 ile son logları kontrol et
- Sırada Faz 5: `epoch_4_lora` veya `best_lora` ile inference (`scripts/run_infer.py`)